In [ ]:
import geopandas as gpd
import h3
import pandas as pd
from shapely.geometry import Point, Polygon


def geocode_kano_incidents(data_list):
    """Converts raw incident lists into a GeoDataFrame.

    In production, replace or augment `location_coords` with an API call
    (e.g., Nominatim / Google Maps API).
    """
    # Pre-mapped coordinates for major Kano hot spots
    kano_landmarks = {
        "Kantin Kwari Market": (11.9961, 8.5283),
        "Kofar Dan Agundi": (11.9868, 8.5147),
        "Sheka Road": (11.9482, 8.5412),
        "Dorayi Quarters": (11.9754, 8.4875),
        "Hotoro": (11.9680, 8.5721),
        "Dala / Kofar Ruwa": (12.0125, 8.5034),
        "Kofar Mata": (11.9912, 8.5321),
        "BUK Road / Matrix Station": (11.9820, 8.4710),
    }

    processed_records = []
    for record in data_list:
        loc = record.get("location_name")
        if loc in kano_landmarks:
            lat, lng = kano_landmarks[loc]
        else:
            # Default fallback for demonstration; replace with geocoding API
            lat, lng = record.get("lat"), record.get("lng")

        processed_records.append(
            {
                "incident_id": record["id"],
                "location_name": loc,
                "timestamp": record["timestamp"],
                "latitude": lat,
                "longitude": lng,
                "geometry": Point(lng, lat),
            }
        )

    df = pd.DataFrame(processed_records)
    return gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")


def assign_h3_hexagons(gdf, resolution=8):
    """Assigns an H3 Spatial Index (Resolution 8 ~ 0.73 km² per hexagon).

    Resolution 8 or 9 is ideal for urban crime neighborhoods.
    """

    def get_h3(row):
        # h3.latlng_to_cell takes (latitude, longitude, resolution)
        return h3.latlng_to_cell(row["latitude"], row["longitude"], resolution)

    gdf["h3_index"] = gdf.apply(get_h3, axis=1)
    return gdf


def aggregate_crime_density(gdf):
    """Aggregates spatial incidents into counts and risk levels per H3 cell."""
    density_df = (
        gdf.groupby("h3_index")
        .size()
        .reset_index(name="incident_count")
    )

    # Classify Risk Level based on count thresholds
    def assign_risk(count):
        if count >= 3:
            return "Extreme"
        elif count == 2:
            return "High"
        else:
            return "Medium"

    density_df["risk_level"] = density_df["incident_count"].apply(assign_risk)

    # Convert H3 Index back into GeoJSON polygons for visualization in Flutter/React Native
    polygons = []
    for cell in density_df["h3_index"]:
        boundary = h3.cell_to_boundary(cell)  # Returns [(lat, lng), ...]
        # GeoJSON expects [(lng, lat), ...]
        flipped_boundary = [(lng, lat) for lat, lng in boundary]
        polygons.append(Polygon(flipped_boundary))

    density_gdf = gpd.GeoDataFrame(
        density_df, geometry=polygons, crs="EPSG:4326"
    )
    return density_gdf


# ==========================================
# Example Execution with Sample Kano Data
# ==========================================
if __name__ == "__main__":
    sample_incidents = [
        {
            "id": "INC-001",
            "location_name": "Sheka Road",
            "timestamp": "2026-03-10 20:30:00",
        },
        {
            "id": "INC-002",
            "location_name": "Sheka Road",
            "timestamp": "2026-03-11 21:00:00",
        },
        {
            "id": "INC-003",
            "location_name": "Sheka Road",
            "timestamp": "2026-03-12 19:45:00",
        },
        {
            "id": "INC-004",
            "location_name": "Hotoro",
            "timestamp": "2026-03-12 22:15:00",
        },
        {
            "id": "INC-005",
            "location_name": "Kantin Kwari Market",
            "timestamp": "2026-03-13 14:10:00",
        },
    ]

    # 1. Geocode
    gdf_incidents = geocode_kano_incidents(sample_incidents)

    # 2. Assign H3 Spatial Index (Resolution 8)
    gdf_indexed = assign_h3_hexagons(gdf_incidents, resolution=8)

    # 3. Aggregate into Spatial Heatmap Data
    density_heatmap = aggregate_crime_density(gdf_indexed)

    print("--- Individual Incident Mapping ---")
    print(gdf_indexed[["incident_id", "location_name", "h3_index"]])

    print("\n--- Aggregated Grid Density (ML Feature Set) ---")
    print(density_heatmap[["h3_index", "incident_count", "risk_level"]])